# H-GNN-RNN-DTI - Objective 1 Fusion Ablation (GPU)

Runs the core RQ1 experiment: **GCA vs concat vs flat_gate vs ungated_xattn** on DAVIS (S1 warm-start), 3 seeds each, reporting AUROC/AUPR mean+/-std + a paired significance test.

The **preprocessed DAVIS dataset is bundled in the repo**, so there is nothing to download and no RDKit / PyTDC to install - only `torch-geometric` on top of Colab's stock PyTorch.

**Before running:** `Runtime -> Change runtime type -> Hardware accelerator = GPU (T4)`, then `Runtime -> Run all`.

Results are checkpointed to Google Drive after every run, so if the session disconnects you can just **Run all again** and it resumes from where it stopped.


## 1. Check the GPU

In [ ]:
!nvidia-smi -L
import torch; print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())

## 2. Get the code (+ bundled preprocessed data)
Re-run safe (wipes any earlier clone) and works for a **private** repo: it tries an anonymous clone first, and only if that fails does it prompt for a GitHub token.

If the repo is private you'll be prompted for a token. Create a short-lived fine-grained token at GitHub -> Settings -> Developer settings -> Fine-grained tokens (Repository access: only `Modeling-DTI`; Permissions: Contents = Read-only), paste it when asked, and revoke it afterwards. The token is read via `getpass` (not echoed, not saved).


In [ ]:
REPO = 'jamcy98/Modeling-DTI'
BRANCH = 'objective1-impl'
import os, shutil, subprocess
from getpass import getpass
os.chdir('/content')  # ensure a valid CWD even if a prior run left us inside a now-deleted clone
DEST = '/content/Modeling-DTI'
if os.path.isdir(DEST):
    shutil.rmtree(DEST)  # remove stale clone from an earlier run

def _clone(url):
    return subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, url, DEST],
                          capture_output=True, text=True)

res = _clone(f'https://github.com/{REPO}.git')  # anonymous (works if public)
if res.returncode != 0:
    print('Private repo - enter a fine-grained GitHub token with Contents:Read on', REPO)
    token = getpass('GitHub token: ').strip()
    res = _clone(f'https://{token}@github.com/{REPO}.git')
    if res.returncode != 0:
        raise RuntimeError('Clone failed:
' + res.stderr[-800:])
%cd /content/Modeling-DTI/H-GNN-RNN-DTI
assert os.path.exists('data/processed/davis/pairs.csv'), 'processed DAVIS missing'
print('bundled DAVIS data found')

## 3. Install dependencies
Colab already ships a CUDA build of PyTorch plus numpy / pandas / scipy / scikit-learn / pyyaml. We only need PyTorch Geometric. (No RDKit or PyTDC: the preprocessed graphs are already in the repo and RDKit is imported lazily.)


In [ ]:
!pip -q install torch-geometric
print('deps installed')

## 4. Mount Google Drive (crash-proof checkpoints)
The ablation writes its results JSON to Drive after every single run. If Colab disconnects, re-open this notebook and Run all again - it skips the runs already on Drive and continues.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
OUT_DIR = '/content/drive/MyDrive/HGNN_RNN_DTI/ablation_results'
os.makedirs(OUT_DIR, exist_ok=True)
print('checkpoints ->', OUT_DIR)

## 5. Run the fusion ablation on GPU
Full DAVIS, all four fusions, seeds 42/43/44. For a fast dry run first, add `--epochs 2 --max-batches 30`.


In [ ]:
!python scripts/run_ablation.py --dataset davis --split S1 --device cuda --seeds 42 43 44 --out-dir "{OUT_DIR}"

## 6. Show + download results

In [ ]:
import json, os
res = json.load(open(os.path.join(OUT_DIR, 'ablation_davis_S1.json')))
print(json.dumps(res['summary'], indent=2))
print(json.dumps(res['significance_gca_vs_baselines'], indent=2))

In [ ]:
from google.colab import files
import os
files.download(os.path.join(OUT_DIR, 'ablation_davis_S1.json'))

## What the result means
- **AUPR** is the honest metric here (DAVIS is ~4% positive).
- RQ1 is supported if **gca** beats concat / flat_gate / ungated_xattn on AUPR, and the paired test `dAUPR` is positive with a small p-value.
- If GCA does *not* beat the simpler fusions, that is a real (publishable) finding too - see docs/objective1_analysis.md Risk R1.

To regenerate the preprocessed data from scratch (needs RDKit + PyTDC): `pip install rdkit 'PyTDC==0.4.1' 'setuptools<81'` then `python scripts/download_data.py --dataset davis && python scripts/preprocess.py --dataset davis`.
